# English → Gujarati NMT — Transformer (optimized, <10h budget)

A from-scratch Transformer encoder–decoder for English→Gujarati, tuned to run inside a
10-hour session on a single 16GB GPU (P100/T4-class).

**What changed vs. earlier drafts, and why:**
- Final layer outputs raw **logits**, loss uses `from_logits=True`. The old version applied
  `softmax` inside the model and used plain `sparse_categorical_crossentropy`. That forces
  TF to materialize a full `(batch, seq_len, vocab)` **float32** probability tensor and its
  gradient as a separate op from the loss — by far the single biggest GPU memory consumer in
  this model, given a 32k-wide vocabulary. Using logits + `SparseCategoricalCrossentropy(from_logits=True)`
  lets TF fuse log-softmax with the loss, avoiding that duplicate full-precision tensor.
- `key_dim = embed_size // num_heads` for every `MultiHeadAttention` layer (not `key_dim = embed_size`).
  The latter silently 8x's the attention projections for no capacity benefit — non-standard sizing.
- Sentences are vectorized to integer ids **once**, up front, into a cached `tf.data` pipeline
  — not re-vectorized from raw strings on every single training step across every epoch.
- Vocabulary is capped (`max_tokens=32_000` per side) via `.adapt()` on the corpus itself, so the
  notebook has no external vocab-file dependency and vocab size is a known, controlled quantity
  rather than whatever an uncapped external file happens to contain.
- `drop_remainder=True` + fixed `max_length` keep batch shapes fully static, which lets XLA
  (`jit_compile=True`) fuse ops instead of bailing out on dynamic shapes.
- Mixed precision (`mixed_float16`) is still on — it's a real, orthogonal memory win on top of
  the fixes above, not a substitute for them.

- A **learning-rate warmup schedule** (from the original Transformer paper), replacing a
  flat LR. At the step counts a single-GPU session can afford, warmup is a major lever for
  getting past mostly-`[UNK]` output -- it changes how the step budget is spent, not how
  much of it exists.
- `ModelCheckpoint` saves every ~1000 **batches**, not just at epoch end, so a session that
  dies mid-epoch still leaves a usable checkpoint.
- Inference (`translate`/`beam_search`) calls the model directly (`model(x, training=False)`)
  instead of `model.predict(x)` in a loop -- `predict()` has real per-call setup overhead
  that adds up across the ~50 calls each translated sentence needs.

**Budget:** this run gets one shot at the remaining GPU quota, so it favors finishing
reliably over squeezing in the absolute maximum data: 1.5M pairs, 5 epochs,
`TimeBudgetStopping` hard-stops at 9h (checkpointing first) leaving real margin under the
quota for setup, vectorization, and the demo/beam-search cells at the end.

## 1. Setup

In [1]:
import numpy as np
import tensorflow as tf
from pathlib import Path

tf.random.set_seed(42)
np.random.seed(42)

# Mixed precision: halves activation/gradient memory for most tensors in the
# model. This is a real win, but it's not what makes the vocab-sized tensors
# tractable -- see the from_logits fix in section 5, which matters more.
tf.keras.mixed_precision.set_global_policy("mixed_float16")

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TF version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Dataset

Download the dataset from Kaggle first (either via the website, or the Kaggle API
if you have `kaggle.json` configured):

```bash
kaggle datasets download -d parvmodi/english-to-gujarati-machine-translation-dataset -p ./data --unzip
```

This gives you `train.en` and `train.gu` inside `./data`. Adjust `DATA_DIR` below
if you put them somewhere else (e.g. `/kaggle/input/...` if running on Kaggle).

In [2]:
DATA_DIR = Path("/kaggle/input/datasets/parvmodi/english-to-gujarati-machine-translation-dataset/en-gu")
# change if needed, e.g. Path("data/en-gu") for a local/Colab run

en_path = "/kaggle/input/datasets/parvmodi/english-to-gujarati-machine-translation-dataset/en-gu/train.en"
gu_path = "/kaggle/input/datasets/parvmodi/english-to-gujarati-machine-translation-dataset/en-gu/train.gu"

with open(en_path, encoding="utf-8") as f:
    sentences_en_raw = [line.strip() for line in f]

with open(gu_path, encoding="utf-8") as f:
    sentences_gu_raw = [line.strip() for line in f]

assert len(sentences_en_raw) == len(sentences_gu_raw), \
    "train.en and train.gu must have the same number of lines"


In [3]:
import re

def clean_gu(text):
    # strip Gujarati danda punctuation marks not covered by default ASCII stripping
    return re.sub(r"[।॥]", "", text).strip()

pairs = list(zip(sentences_en_raw, sentences_gu_raw))
np.random.shuffle(pairs)

sentences_en, sentences_gu = zip(*pairs)
sentences_gu = [clean_gu(s) for s in sentences_gu]

print(f"{len(sentences_en):,} shuffled pairs")
for i in range(3):
    print(sentences_en[i], "=>", sentences_gu[i])


3,067,790 shuffled pairs
This doesnt require a lot of effort. => તે માટે બહુ મહેનતની પણ જરૂર નથી.
Clean the doors. => દરવાજા અને બારીઓ સાફ કરો.
Jones was part of Star Sports commentary team and was in a bio-secure bubble in a seven-star hotel in Mumbai. => જોન્સ સ્ટાર સ્પોર્ટ્સની કોમેન્ટ્રી ટીમનો ભાગ હતા અને મુંબઈની સેવન સ્ટાર હોટલમાં રોકાયા હતા.


In [4]:
# This run gets ONE shot at the remaining GPU quota, so this trades a bit of
# raw data volume for a much higher chance of actually finishing with a good
# checkpoint. 1.5M pairs (down from the full ~2.6-3M) means more epochs (more
# gradient updates -- see the LR schedule note in section 6) complete within
# budget, and the first checkpoint lands sooner.
SUBSAMPLE_SIZE = 1_500_000

if SUBSAMPLE_SIZE is not None:
    sentences_en = sentences_en[:SUBSAMPLE_SIZE]
    sentences_gu = sentences_gu[:SUBSAMPLE_SIZE]

print(f"{len(sentences_en):,} pairs in use")


1,500,000 pairs in use


## 3. Vocabulary

Built directly from the corpus via `.adapt()` rather than an external vocab file, so
vocab size is a known, controlled quantity. Capped at 32k tokens per side -- large
enough to keep `<unk>` rates low, small enough that the final Dense layer and its
output tensor (the model's actual memory bottleneck -- see section 5) don't balloon.

In [5]:
max_length = 50
MAX_VOCAB = 32_000

text_vec_layer_en = tf.keras.layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_sequence_length=max_length,
)
text_vec_layer_gu = tf.keras.layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_sequence_length=max_length,
)

text_vec_layer_en.adapt(sentences_en)
text_vec_layer_gu.adapt([f"startofseq {s} endofseq" for s in sentences_gu])

vocab_size_en = text_vec_layer_en.vocabulary_size()
vocab_size_gu = text_vec_layer_gu.vocabulary_size()

print("English vocab size:", vocab_size_en)
print("Gujarati vocab size:", vocab_size_gu)
print(text_vec_layer_en.get_vocabulary()[:10])
print(text_vec_layer_gu.get_vocabulary()[:10])


I0000 00:00:1786355848.180483      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


English vocab size: 32000
Gujarati vocab size: 32000
['', '[UNK]', np.str_('the'), np.str_('of'), np.str_('and'), np.str_('to'), np.str_('in'), np.str_('a'), np.str_('is'), np.str_('for')]
['', '[UNK]', np.str_('startofseq'), np.str_('endofseq'), np.str_('છે'), np.str_('અને'), np.str_('આ'), np.str_('પણ'), np.str_('માટે'), np.str_('પર')]


In [6]:
vocab = set(text_vec_layer_gu.get_vocabulary())
all_tokens = " ".join(sentences_gu).split()
total = len(all_tokens)
oov = sum(1 for t in all_tokens if t not in vocab)
print(f"OOV rate: {oov/total:.2%}  ({oov:,} / {total:,} tokens)")


OOV rate: 20.75%  (2,963,525 / 14,279,164 tokens)


## 4. Pre-vectorize once, build a `tf.data` pipeline

The earlier version kept `TextVectorization` as the model's first layer and fed it raw
strings, meaning every one of the ~25k+ training steps re-ran string vectorization on
that step's batch. Vectorizing the whole corpus to int32 ids **once** here and caching
it removes that repeated work from the training loop, and int32 id tensors are cheaper
to move host→device than variable-length string tensors.

`drop_remainder=True` keeps every batch's shape identical, which is what lets
`jit_compile=True` (section 6) actually fuse the graph instead of falling back to a
dynamic-shape path.

In [7]:
BATCH_SIZE = 256

n_train = int(0.85 * len(sentences_en))

X_train_txt, X_valid_txt = sentences_en[:n_train], sentences_en[n_train:]
dec_in_train_txt = [f"startofseq {s}" for s in sentences_gu[:n_train]]
dec_in_valid_txt = [f"startofseq {s}" for s in sentences_gu[n_train:]]
dec_out_train_txt = [f"{s} endofseq" for s in sentences_gu[:n_train]]
dec_out_valid_txt = [f"{s} endofseq" for s in sentences_gu[n_train:]]

# Vectorize once. TextVectorization batches internally fine on a full list this size
# (a few million short strings) -- this is a single pass, not a per-step cost.
X_train_ids = text_vec_layer_en(tf.constant(X_train_txt))
X_valid_ids = text_vec_layer_en(tf.constant(X_valid_txt))
Xd_train_ids = text_vec_layer_gu(tf.constant(dec_in_train_txt))
Xd_valid_ids = text_vec_layer_gu(tf.constant(dec_in_valid_txt))
Y_train = text_vec_layer_gu(tf.constant(dec_out_train_txt))
Y_valid = text_vec_layer_gu(tf.constant(dec_out_valid_txt))

print("train:", X_train_ids.shape, "valid:", X_valid_ids.shape)

train_ds = (
    tf.data.Dataset.from_tensor_slices(((X_train_ids, Xd_train_ids), Y_train))
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)
valid_ds = (
    tf.data.Dataset.from_tensor_slices(((X_valid_ids, Xd_valid_ids), Y_valid))
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)


train: (1275000, 50) valid: (225000, 50)


In [8]:
import pickle
from pathlib import Path

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def save_artifacts(model, name):
    """Save a trained model (.keras) and the shared TextVectorization
    vocabularies, so later inference (e.g. a demo website) can just load
    these files instead of retraining or re-running adapt()."""
    model_path = CHECKPOINT_DIR / f"{name}.keras"
    model.save(model_path)

    vocab_path = CHECKPOINT_DIR / "vectorizer_vocab.pkl"
    if not vocab_path.exists():
        with open(vocab_path, "wb") as f:
            pickle.dump({
                "en_vocab": text_vec_layer_en.get_vocabulary(),
                "gu_vocab": text_vec_layer_gu.get_vocabulary(),
                "vocab_size_en": vocab_size_en,
                "vocab_size_gu": vocab_size_gu,
                "max_length": max_length,
            }, f)

    print(f"Saved {model_path}")


## 5. Model

Same encoder–decoder Transformer shape as before (`embed_size=128`, `N=2` blocks,
`num_heads=8`), with two structural fixes:

1. Inputs are `int32` token ids (`shape=[max_length]`), not raw strings -- vectorization
   already happened in section 4, so it isn't part of the model graph or the training step.
2. `key_dim = embed_size // num_heads` on every `MultiHeadAttention` call -- correct,
   standard Transformer sizing (`key_dim=embed_size` was inflating every attention layer
   ~8x for no capacity benefit).
3. The final `Dense` layer has **no activation** -- it outputs logits. Softmax is fused
   into the loss instead of being a separate materialized tensor (see section 6).

In [9]:
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, max_length, embed_size, dtype=tf.float32, **kwargs):
        super().__init__(dtype=dtype, **kwargs)
        assert embed_size % 2 == 0, "embed_size must be even"
        self.max_length = max_length
        self.embed_size = embed_size
        p, i = np.meshgrid(np.arange(max_length), 2 * np.arange(embed_size // 2))
        pos_emb = np.empty((1, max_length, embed_size))
        pos_emb[0, :, ::2] = np.sin(p / 10_000 ** (i / embed_size)).T
        pos_emb[0, :, 1::2] = np.cos(p / 10_000 ** (i / embed_size)).T
        self.pos_encodings = tf.constant(pos_emb.astype(self.dtype))
        self.supports_masking = True

    def call(self, inputs):
        batch_max_length = tf.shape(inputs)[1]
        return inputs + self.pos_encodings[:, :batch_max_length]

    def get_config(self):
        config = super().get_config()
        config.update({"max_length": self.max_length, "embed_size": self.embed_size})
        return config


In [10]:
embed_size = 128

encoder_input_ids = tf.keras.layers.Input(shape=[max_length], dtype=tf.int32, name="encoder_ids")
decoder_input_ids = tf.keras.layers.Input(shape=[max_length], dtype=tf.int32, name="decoder_ids")

encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size_en, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size_gu, embed_size, mask_zero=True)

encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

pos_embed_layer = PositionalEncoding(max_length, embed_size)
encoder_in = pos_embed_layer(encoder_embeddings)
decoder_in = pos_embed_layer(decoder_embeddings)


In [11]:
N = 2
num_heads = 8
key_dim = embed_size // num_heads   # 16 -- standard sizing, not embed_size
dropout_rate = 0.1
n_units = 256

encoder_pad_mask = tf.keras.ops.not_equal(encoder_input_ids, 0)[:, tf.newaxis]

batch_max_len_dec = tf.keras.ops.shape(decoder_embeddings)[1]
decoder_pad_mask = tf.keras.ops.not_equal(decoder_input_ids, 0)[:, tf.newaxis]
causal_mask = tf.keras.ops.cast(
    tf.keras.ops.tril(tf.keras.ops.ones((batch_max_len_dec, batch_max_len_dec))),
    "bool"
)


In [12]:
Z = encoder_in
for _ in range(N):
    skip = Z
    attn_layer = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim, dropout=dropout_rate)
    Z = attn_layer(Z, value=Z, attention_mask=encoder_pad_mask)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))

    skip = Z
    Z = tf.keras.layers.Dense(n_units, activation="relu")(Z)
    Z = tf.keras.layers.Dense(embed_size)(Z)
    Z = tf.keras.layers.Dropout(dropout_rate)(Z)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))

encoder_outputs = Z


In [13]:
Z = decoder_in
for _ in range(N):
    skip = Z
    attn_layer = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim, dropout=dropout_rate)
    Z = attn_layer(Z, value=Z, attention_mask=causal_mask & decoder_pad_mask)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))

    skip = Z
    attn_layer = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim, dropout=dropout_rate)
    Z = attn_layer(Z, value=encoder_outputs, attention_mask=encoder_pad_mask)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))

    skip = Z
    Z = tf.keras.layers.Dense(n_units, activation="relu")(Z)
    Z = tf.keras.layers.Dense(embed_size)(Z)
    Z = tf.keras.layers.Dropout(dropout_rate)(Z)   # matches encoder FFN block now
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))


## 6. Compile: logits + fused loss + LR warmup

Two independent fixes here, one for memory, one for quality:

- **No `activation="softmax"`** on the output layer --
  `SparseCategoricalCrossentropy(from_logits=True)` computes log-softmax and the loss
  together instead of materializing a separate full-precision `(batch, seq_len, vocab)`
  probability tensor plus its gradient as distinct ops. `SparseCategoricalAccuracy` works
  fine directly on logits (argmax is unaffected by softmax's monotonicity).
- **A warmup + inverse-sqrt-decay learning rate schedule**, the one from the original
  Transformer paper, instead of a flat learning rate. This is the main lever for the
  "mostly `[UNK]`" output from the earlier 800k run: at ~16k total gradient steps with a
  fixed LR, the model may not have found a good region of the loss surface at all yet.
  Warmup (LR ramps up over the first `warmup_steps`, then decays) is specifically known
  to matter for Transformers early in training and costs nothing extra compute-wise --
  it changes *how* the existing step budget is spent, not how much of it there is.

`jit_compile=True` turns on XLA for the train step -- safe here because every batch has
a static shape (`drop_remainder=True` + fixed `max_length`).

In [14]:
class TransformerSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Warmup then inverse-sqrt decay, as in 'Attention Is All You Need' (section 5.3).
    LR ramps linearly for `warmup_steps`, then decays as 1/sqrt(step). Costs nothing
    extra -- it changes how the step budget is spent, which matters a lot when total
    steps are in the tens of thousands rather than hundreds of thousands."""
    def __init__(self, embed_size, warmup_steps=4000):
        super().__init__()
        self.embed_size = tf.cast(embed_size, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.embed_size) * tf.minimum(arg1, arg2)

    def get_config(self):
        return {"embed_size": int(self.embed_size.numpy()), "warmup_steps": self.warmup_steps}

lr_schedule = TransformerSchedule(embed_size, warmup_steps=4000)
# beta_2=0.98, epsilon=1e-9 -- also from the paper; the default Adam epsilon (1e-7)
# is fine too, but this pairing is what the schedule above was tuned against.
optimizer = tf.keras.optimizers.Adam(lr_schedule, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

Y_logits = tf.keras.layers.Dense(vocab_size_gu, dtype="float32")(Z)  # logits, no activation

transformer_model = tf.keras.Model(
    inputs=[encoder_input_ids, decoder_input_ids], outputs=[Y_logits])

transformer_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    jit_compile=True,
)
transformer_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ decoder_ids         │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_ids         │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 50, 128)   │  4,096,000 │ decoder_ids[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 50)        │          0 │ encoder_ids[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 50, 128)   │  4,096,000 │ encoder_ids[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encoding │ (None, 50, 128)   │          0 │ embedding[0][0],  │
│ (PositionalEncodin… │                   │            │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 1, 50)     │          0 │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 128)   │     66,048 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ get_item[0][0],   │
│                     │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 128)   │          0 │ multi_head_atten… │
│                     │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 50, 128)   │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 256)   │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 50, 128)   │     32,896 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 50, 128)   │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 50, 128)   │          0 │ dropout_1[0][0],  │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 128)   │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 128)   │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ get_item[0][0],   │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 50, 128)   │          0 │ multi_head_atten… │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 12,982,528 (49.52 MB)

 Trainable params: 12,982,528 (49.52 MB)

 Non-trainable params: 0 (0.00 B)

## 7. Train, with a hard wall-clock stop AND frequent checkpoints

Two safety nets, because this run only gets one shot at the remaining quota:

- `TimeBudgetStopping` checkpoints and stops cleanly at the budget so the session can't
  blow past what's left -- it saves progress rather than losing it.
- `ModelCheckpoint` below saves every `save_freq` **batches**, not just at epoch end.
  With `save_freq="epoch"`, a session that dies mid-epoch (disconnect, quota cutoff,
  anything) saves nothing at all -- exactly what happened on the last attempt. Saving
  every ~1000 steps means the worst case is losing a few minutes of progress, not the
  whole epoch.

Budget is set to 9h, leaving ~1h45m buffer under the ~10h49m quota for data
loading/vectorization (a few minutes) and the demo/beam-search cells at the end.

In [15]:
import time

TRAIN_TIME_BUDGET_SECONDS = 9 * 3600  # 9h training budget, ~1h45m buffer under quota

class TimeBudgetStopping(tf.keras.callbacks.Callback):
    """Stops training gracefully once wall-clock time since .fit() began
    exceeds budget_seconds, saving the model first so partial progress from a
    session that would otherwise blow the time budget is not lost."""
    def __init__(self, budget_seconds, checkpoint_path, check_every_n_batches=200):
        super().__init__()
        self.budget_seconds = budget_seconds
        self.checkpoint_path = checkpoint_path
        self.check_every_n_batches = check_every_n_batches
        self.start_time = None

    def on_train_begin(self, logs=None):
        self.start_time = time.time()

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        elapsed = time.time() - self.start_time
        epoch_dur = time.time() - self._epoch_start
        print(f"Epoch {epoch + 1} took {epoch_dur / 60:.1f} min "
              f"({elapsed / 3600:.2f}h elapsed of {self.budget_seconds / 3600:.1f}h budget)")

    def on_train_batch_end(self, batch, logs=None):
        if batch % self.check_every_n_batches != 0:
            return
        elapsed = time.time() - self.start_time
        if elapsed > self.budget_seconds:
            print(f"\nTime budget ({self.budget_seconds / 3600:.1f}h) reached at "
                  f"{elapsed / 3600:.2f}h elapsed -- stopping training and saving "
                  f"progress now to stay under budget.")
            self.model.stop_training = True
            self.model.save(self.checkpoint_path)

time_budget_cb = TimeBudgetStopping(
    TRAIN_TIME_BUDGET_SECONDS, CHECKPOINT_DIR / "transformer_model.keras")


In [16]:
history_transformer = transformer_model.fit(
    train_ds,
    epochs=5,
    # 5 epochs over 1.5M pairs -- with the LR schedule doing more with each step,
    # more epochs at this size beat fewer epochs over the full dataset here. Watch
    # the first epoch's timing print: if you're comfortably under budget, this will
    # finish naturally; if not, TimeBudgetStopping and the step-checkpoints below
    # both have you covered.
    validation_data=valid_ds,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            CHECKPOINT_DIR / "transformer_model.keras", save_freq=1000),
        time_budget_cb,
    ],
)

transformer_model.save(CHECKPOINT_DIR / "transformer_model.keras")
print("Model saved to", CHECKPOINT_DIR / "transformer_model.keras")


Epoch 1/5
   1/4980 ━━━━━━━━━━━━━━━━━━━━ 43:51:42 32s/step - accuracy: 0.0000e+00 - loss: 10.3002

I0000 00:00:1786355926.462034      67 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


4980/4980 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 0.7790 - loss: 3.2949Epoch 1 took 9.9 min (0.16h elapsed of 9.0h budget)
4980/4980 ━━━━━━━━━━━━━━━━━━━━ 594s 113ms/step - accuracy: 0.8351 - loss: 1.6506 - val_accuracy: 0.8671 - val_loss: 0.8141
Epoch 2/5
4980/4980 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.8685 - loss: 0.7892Epoch 2 took 9.4 min (0.32h elapsed of 9.0h budget)
4980/4980 ━━━━━━━━━━━━━━━━━━━━ 565s 113ms/step - accuracy: 0.8710 - loss: 0.7606 - val_accuracy: 0.8753 - val_loss: 0.7161
Epoch 3/5
4980/4980 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.8757 - loss: 0.7071Epoch 3 took 9.4 min (0.48h elapsed of 9.0h budget)
4980/4980 ━━━━━━━━━━━━━━━━━━━━ 566s 114ms/step - accuracy: 0.8767 - loss: 0.6975 - val_accuracy: 0.8781 - val_loss: 0.6886
Epoch 4/5
4980/4980 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.8788 - loss: 0.6762Epoch 4 took 9.4 min (0.64h elapsed of 9.0h budget)
4980/4980 ━━━━━━━━━━━━━━━━━━━━ 565s 114ms/step - accuracy: 0.8794 - loss: 0.6711 

In [17]:
save_artifacts(transformer_model, "transformer_model")


Saved /kaggle/working/checkpoints/transformer_model.keras


## 8. Translate

Inference takes raw strings, vectorizes with the fitted layers, and calls the model --
the model itself no longer does vectorization internally. Logits, not probabilities, come
back; `argmax` is unaffected by softmax (it's monotonic), so greedy decoding needs no
softmax at all. Beam search still needs comparable per-step scores, so it applies
`log_softmax` to just that step's `(vocab,)` logits vector -- cheap, since it's a single
row, not a full-batch tensor.

**`model(x, training=False)` instead of `model.predict(x)`.** Each of these loops calls
the model up to `max_length` times per sentence. `predict()` sets up a `tf.data` pipeline
on every single call -- fine once, wasteful 50+ times in a tight Python loop. Direct calls
are the documented faster path for exactly this pattern and return the same values.

In [18]:
def translate(sentence_en, model=None, vec_layer_out=None):
    model = model or transformer_model
    vec_layer_out = vec_layer_out or text_vec_layer_gu
    translation = ""
    for word_idx in range(max_length):
        X = text_vec_layer_en(tf.constant([sentence_en]))
        X_dec = vec_layer_out(tf.constant(["startofseq " + translation]))
        y_logits = model((X, X_dec), training=False).numpy()[0, word_idx]
        predicted_word_id = np.argmax(y_logits)  # argmax is softmax-invariant
        predicted_word = vec_layer_out.get_vocabulary()[predicted_word_id]
        if predicted_word == "endofseq":
            break
        translation += " " + predicted_word
    return translation.strip()


In [19]:
print(translate("I like soccer", model=transformer_model))
print(translate("I like soccer and also going to the beach", model=transformer_model))
print(translate("Would you like to swap jobs?", model=transformer_model))


હું ફૂટબોલ જેવી છું
હું ફૂટબોલ અને [UNK] પણ [UNK]
શું તમે [UNK] નોકરી પસંદ કરો છો


## 9. Beam search decoding

In [20]:
def beam_search(sentence_en, model=None, vec_layer_out=None, beam_width=3, verbose=False):
    model = model or transformer_model
    vec_layer_out = vec_layer_out or text_vec_layer_gu
    vocab = vec_layer_out.get_vocabulary()
    eos_id = vocab.index("endofseq")

    X = text_vec_layer_en(tf.constant([sentence_en]))
    X_dec = vec_layer_out(tf.constant(["startofseq"]))
    y_logits = model((X, X_dec), training=False).numpy()[0, 0]
    y_logp = tf.nn.log_softmax(y_logits).numpy()  # single row -- cheap
    top_k = np.argsort(-y_logp)[:beam_width]
    candidates = [(y_logp[idx], [idx], idx == eos_id) for idx in top_k]

    for step in range(1, max_length):
        all_candidates = []
        for log_prob, token_ids, finished in candidates:
            if finished:
                all_candidates.append((log_prob, token_ids, True))
                continue
            translation = " ".join(vocab[t] for t in token_ids if t not in (0, eos_id))
            X_dec = vec_layer_out(tf.constant(["startofseq " + translation]))
            y_logits = model((X, X_dec), training=False).numpy()[0, step]
            y_logp = tf.nn.log_softmax(y_logits).numpy()
            top_k = np.argsort(-y_logp)[:beam_width]
            for idx in top_k:
                new_log_prob = log_prob + y_logp[idx]
                all_candidates.append((new_log_prob, token_ids + [idx], idx == eos_id))

        def score(c):
            log_prob, token_ids, _ = c
            return log_prob / len(token_ids)

        candidates = sorted(all_candidates, key=score, reverse=True)[:beam_width]
        if verbose:
            print(f"step {step}: " + " | ".join(
                " ".join(vocab[t] for t in c[1] if t not in (0, eos_id)) for c in candidates))
        if all(c[2] for c in candidates):
            break

    best_log_prob, best_tokens, _ = max(candidates, key=score)
    words = [vocab[t] for t in best_tokens if t not in (0, eos_id)]
    return " ".join(words)

print(beam_search("I like soccer and also going to the beach", model=transformer_model))


મને ફૂટબોલ ગમે છે અને બીચ પણ જાય છે


## 10. Next steps

- If `<unk>` still dominates output, word-level tokenization has a real ceiling on this
  dataset -- subword (BPE/SentencePiece) is the next lever, not a bigger word vocab.
- Track BLEU on a held-out split rather than eyeballing translations.
- The trained model is saved to `/kaggle/working/checkpoints/transformer_model.keras`,
  and `vectorizer_vocab.pkl` has both vocabularies -- save this as a Kaggle version and
  they'll be in the Output tab, ready for inference without retraining.
- Watch the epoch-timing prints from the training run. If epochs are landing comfortably
  under budget, there's room to try a larger `BATCH_SIZE` (e.g. 384) next run -- the
  from_logits fix freed real memory headroom that a bigger batch could use for better
  GPU utilization and fewer total steps.
- If output is still mostly `[UNK]`/garbled after this run, that's now more likely a real
  step-count ceiling than a missing-fix problem -- word-level tokenization and subword
  (BPE/SentencePiece) are the next lever, per the OOV-rate check in section 3.
- Inference in `translate`/`beam_search` re-encodes the source sentence and re-runs the
  full decoder stack from scratch at every single output token (no KV-caching). Fine for
  occasional demo calls; if this becomes a real serving path, that's the next thing to fix,
  not the training loop.